# 04 - Models: Linear Regression con TF-IDF

Este notebook entrena un modelo de **Linear Regression** para la familia `04_models_<miembro>` usando la codificacion **TF-IDF** generada en `02_encoding.ipynb`.

Como `LinearRegression` produce una salida continua, para evaluar clasificacion binaria se umbraliza la prediccion en `0.5` y se recorta el score al rango `[0, 1]` cuando se calcula `ROC AUC`. El flujo mantiene el mismo patron de evaluacion y registro en **MLflow** del notebook `03_baseline.ipynb`.

## 1. Instalacion e imports

In [ ]:
!pip install -q scikit-learn scipy joblib matplotlib seaborn mlflow

In [ ]:
import logging
import warnings

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.sparse import load_npz
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

SEED = 42
MEMBER = "asantiagodiaz"
MLFLOW_URI = "http://ec2-52-5-36-177.compute-1.amazonaws.com:5000"

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("deep")
np.random.seed(SEED)

## 2. Carga de artefactos

In [ ]:
X_tr = load_npz('X_tr.npz')
X_val = load_npz('X_val.npz')
X_train_tfidf = load_npz('X_train_tfidf.npz')
X_test_tfidf = load_npz('X_test_tfidf.npz')

y_tr = joblib.load('y_tr.pkl')
y_val = joblib.load('y_val.pkl')
y_train = joblib.load('y_train.pkl')
y_test = joblib.load('y_test.pkl')

print('Shapes cargados:')
print('  X_tr          :', X_tr.shape)
print('  X_val         :', X_val.shape)
print('  X_train_tfidf :', X_train_tfidf.shape)
print('  X_test_tfidf  :', X_test_tfidf.shape)
print('  y_tr          :', len(y_tr))
print('  y_val         :', len(y_val))
print('  y_train       :', len(y_train))
print('  y_test        :', len(y_test))

## 3. Entrenamiento con todo el dataset

Este notebook usa **todo el dataset disponible en los artefactos de `02_encoding.ipynb`**. `X_tr` se usa completo para seleccionar hiperparametros, `X_val` completo para validacion, y el modelo final se reentrena sobre `X_train_tfidf` completo.

In [ ]:
print('Configuracion de entrenamiento:')
print('  Tuning train :', X_tr.shape)
print('  Validacion   :', X_val.shape)
print('  Train final  :', X_train_tfidf.shape)
print('  Test final   :', X_test_tfidf.shape)
print('  balance y_tr   :', np.bincount(y_tr))
print('  balance y_val  :', np.bincount(y_val))
print('  balance y_train:', np.bincount(y_train))
print('  balance y_test :', np.bincount(y_test))

## 4. Busqueda de hiperparametros

Se entrena `LinearRegression` directamente sobre la matriz sparse TF-IDF y se selecciona la mejor configuracion por `f1_macro` en validacion despues de aplicar umbral `0.5` sobre la salida continua.

In [ ]:
THRESHOLD = 0.5

param_grid = [
    {'fit_intercept': True},
    {'fit_intercept': False},
]

def get_predictions_from_scores(scores, threshold=THRESHOLD):
    scores = np.asarray(scores).reshape(-1)
    scores_clipped = np.clip(scores, 0, 1)
    preds = (scores_clipped >= threshold).astype(int)
    return scores_clipped, preds

results_lin = []
best_model = None
best_params = None
best_f1 = -1

for params in param_grid:
    model = LinearRegression(fit_intercept=params['fit_intercept'])

    model.fit(X_tr, y_tr)
    y_val_score_raw = model.predict(X_val)
    y_val_score, y_val_pred = get_predictions_from_scores(y_val_score_raw)

    metrics_row = {
        **params,
        'accuracy': round(accuracy_score(y_val, y_val_pred), 4),
        'f1_macro': round(f1_score(y_val, y_val_pred, average='macro'), 4),
        'precision_macro': round(precision_score(y_val, y_val_pred, average='macro'), 4),
        'recall_macro': round(recall_score(y_val, y_val_pred, average='macro'), 4),
        'roc_auc': round(roc_auc_score(y_val, y_val_score), 4),
    }
    results_lin.append(metrics_row)

    if metrics_row['f1_macro'] > best_f1:
        best_f1 = metrics_row['f1_macro']
        best_params = params
        best_model = model

df_lin = pd.DataFrame(results_lin).sort_values('f1_macro', ascending=False).reset_index(drop=True)
display(df_lin)
print('Mejores hiperparametros:', best_params)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = df_lin.copy()
plot_df['config'] = [f"fit_intercept={r.fit_intercept}" for r in plot_df.itertuples()]
ax.bar(plot_df['config'], plot_df['f1_macro'], color='steelblue')
ax.set_title('Linear Regression - F1 macro en validacion')
ax.set_xlabel('Configuracion')
ax.set_ylabel('F1 macro')
plt.tight_layout()
plt.show()

## 5. Entrenamiento final

Se reentrena el mejor modelo usando **todo `X_train_tfidf`** y sus etiquetas `y_train`.

In [ ]:
X_train_final = X_train_tfidf
y_train_final = y_train

model_final = LinearRegression(fit_intercept=best_params['fit_intercept'])

model_final.fit(X_train_final, y_train_final)

print('Modelo final entrenado con todo el train.')
print('  Train final shape:', X_train_final.shape)
print('  fit_intercept   :', best_params['fit_intercept'])
print('  threshold       :', THRESHOLD)

## 6. Evaluacion final en test

In [ ]:
y_score_raw = model_final.predict(X_test_tfidf)
y_score, y_pred = get_predictions_from_scores(y_score_raw)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')
auc = roc_auc_score(y_test, y_score)
precision = precision_score(y_test, y_pred, average='macro')
recall = recall_score(y_test, y_pred, average='macro')

print('Metricas en test:')
print(f'  Accuracy   : {acc:.4f}')
print(f'  F1 macro   : {f1:.4f}')
print(f'  Precision  : {precision:.4f}')
print(f'  Recall     : {recall:.4f}')
print(f'  ROC AUC    : {auc:.4f}')
print()
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negativo (0)', 'Positivo (1)'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Linear Regression + TF-IDF')
plt.tight_layout()
plt.show()

## 7. Guardar modelo y resultados

In [ ]:
joblib.dump(model_final, 'linear_regression_tfidf_model.pkl')

results_final = pd.DataFrame([{
    'modelo': 'LinearRegression',
    'encoding': 'TF-IDF',
    'member': MEMBER,
    'train_size_tuning': X_tr.shape[0],
    'val_size': X_val.shape[0],
    'final_train_size': len(y_train_final),
    'fit_intercept': best_params['fit_intercept'],
    'threshold': THRESHOLD,
    'accuracy': round(acc, 4),
    'f1_macro': round(f1, 4),
    'roc_auc': round(auc, 4)
}])

results_final.to_csv('results_linear_regression_tfidf.csv', index=False)

print('Archivos guardados:')
print('  linear_regression_tfidf_model.pkl -> modelo Linear Regression + TF-IDF')
print('  results_linear_regression_tfidf.csv -> tabla de metricas')
print()
print(results_final.to_string(index=False))

## 8. Registro en MLflow

In [ ]:
import logging
import warnings

logging.getLogger('mlflow.tracking.request_header.registry').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', message='.*Saving scikit-learn models in the pickle or cloudpickle format.*', category=FutureWarning)

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment('Parcial_1_NLP')

with mlflow.start_run(run_name=f'LinearRegression_TFIDF_{MEMBER}') as run:
    mlflow.set_tags({
        'user': 'Santiago Diaz',
        'member': MEMBER,
        'model_type': 'LinearRegression',
        'encoding': 'TF-IDF',
        'dataset': 'Sentiment140Twitter',
    })

    mlflow.log_params({
        'prep_remove_urls': True,
        'prep_remove_mentions': True,
        'prep_remove_hashtags': True,
        'prep_remove_emojis': True,
        'prep_remove_stopwords': False,
        'prep_lemmatization': False,
    })

    mlflow.log_params({
        'vec_type': 'TfidfVectorizer',
        'vec_max_features': 100000,
        'vec_min_df': 5,
        'vec_max_df': 0.95,
        'vec_ngram_range': '(1,2)',
        'vec_sublinear_tf': True,
    })

    mlflow.log_params({
        'model': 'LinearRegression',
        'fit_intercept': best_params['fit_intercept'],
        'threshold': THRESHOLD,
        'train_size_tuning': X_tr.shape[0],
        'val_size': X_val.shape[0],
        'final_train_size': len(y_train_final),
        'test_size': X_test_tfidf.shape[0],
        'vocab_size': X_train_tfidf.shape[1],
    })

    y_score_train_raw = model_final.predict(X_train_final)
    y_score_train, y_pred_train = get_predictions_from_scores(y_score_train_raw)
    mlflow.log_metrics({
        'train_accuracy': round(accuracy_score(y_train_final, y_pred_train), 4),
        'train_f1_macro': round(f1_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_precision': round(precision_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_recall': round(recall_score(y_train_final, y_pred_train, average='macro'), 4),
        'train_roc_auc': round(roc_auc_score(y_train_final, y_score_train), 4),
    })

    y_score_val_raw = best_model.predict(X_val)
    y_score_val, y_pred_val_f = get_predictions_from_scores(y_score_val_raw)
    mlflow.log_metrics({
        'val_accuracy': round(accuracy_score(y_val, y_pred_val_f), 4),
        'val_f1_macro': round(f1_score(y_val, y_pred_val_f, average='macro'), 4),
        'val_precision': round(precision_score(y_val, y_pred_val_f, average='macro'), 4),
        'val_recall': round(recall_score(y_val, y_pred_val_f, average='macro'), 4),
        'val_roc_auc': round(roc_auc_score(y_val, y_score_val), 4),
    })

    mlflow.log_metrics({
        'test_accuracy': round(acc, 4),
        'test_f1_macro': round(f1, 4),
        'test_precision': round(precision, 4),
        'test_recall': round(recall, 4),
        'test_roc_auc': round(auc, 4),
    })

    mlflow.log_artifact('results_linear_regression_tfidf.csv')
    mlflow.sklearn.log_model(
        model_final,
        name='model',
        registered_model_name=f'LinearRegression_TFIDF_{MEMBER}'
    )

    print('=' * 55)
    print('  Run registrado en MLflow')
    print(f'  Servidor    : {MLFLOW_URI}')
    print('  Experimento : Parcial_1_NLP')
    print(f'  Corrida     : LinearRegression_TFIDF_{MEMBER}')
    print(f'  Run ID      : {run.info.run_id}')
    print(f'  Test F1     : {f1:.4f}')
    print(f'  Test AUC    : {auc:.4f}')
    print('=' * 55)